# Análisis Hidrológico — Tributarios del Río Cauca
### Proyecto Corredor Biológico 890K · UAO – ASOCAÑA · Fase 1 Diagnóstico

**Estructura de carpetas esperada:**
```
Tributarios/
├── AMAIME/
│   ├── AMAIME (Hasta 2023)/
│   │   └── Informe.html
│   └── LOS CEIBOS (Desde 2021)/
│       └── Informe.html
├── GUABAS/PUENTE PIEDRA/Informe.html
├── GUADALAJARA/EL VERGEL/Informe.html
├── RIO BOLO/ARRIBA (S - HASTA 2022)/Informe.html
├── RIO BOLO/LOS MINCHOS (HASTA 2023)/Informe.html
├── RIO BUGALAGRANDE/EL PLACER/Informe.html
├── RIO DESBARATADO/ORTIGAL (HASTA 2023 - S)/Informe.html
├── RIO FRAILE/BUICHITOLO (HASTA 2022)/Informe.html
├── RIO FRAILE/LA INDUSTRIA/Informe.html
├── RIO LA PAILA/LA SORPRESA/Informe.html
├── RIO PALO/BOCATOMA (HASTA 2024)/Informe.html
├── RIO PALO/CANAL BOCATOMA (HASTA 2022)/Informe.html
├── RIO PALO/PUERTO TEJADA/Informe.html
├── RIOFRIO/SALONICA/Informe.html
├── TULUA/LA RAFAELA/Informe (Hasta 2016).html
├── TULUA/MATEGUADUA/Informe.html
└── RIO RISARALDA/
    └── CAUDAL/
        ├── Rio_Risaralda_caudal.xlsx
        └── Casa_Maquinas_Rda_caudal.xlsx
```

**Productos generados por celda:**
| Celda | Producto |
|---|---|
| 1 | Configuración de rutas |
| 2 | Extrae HTML → `caudal.xlsx` por estación |
| 3 | Curva de duración de caudales + umbrales |
| 4 | Caudal promedio mensual por quinquenio |
| 5 | Caudal promedio anual |
| 6 | Curva de duración — Río Risaralda (XLSX propios) |

> **Ríos sin datos de caudal:** Nima, Zabaletas, Guachal — no tienen estaciones en el portal CVC.


## Celda 1 · Configuración de rutas
Edita `RAIZ_TRIBUTARIOS` para que apunte a la carpeta `Tributarios/` en tu equipo.

In [1]:
# ── Dependencias ──────────────────────────────────────────────────────────────
# pip install beautifulsoup4 lxml openpyxl matplotlib seaborn numpy pandas
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from bs4 import BeautifulSoup
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  AJUSTA ESTA RUTA a la ubicación real de la carpeta Tributarios/ en     │
# │  tu equipo.  Ejemplos:                                                   │
# │      Windows : Path(r"C:\Users\saulo\Desktop\Tributarios")           │
# │      macOS   : Path("/Users/saulo/Desktop/Tributarios")                 │
# │      Linux   : Path("/home/saulo/proyecto/Tributarios")                 │
# └─────────────────────────────────────────────────────────────────────────┘
RAIZ_TRIBUTARIOS = Path("Tributarios")          # ← editar si es necesario

# Constantes globales
MESES         = ["ENE","FEB","MAR","ABR","MAY","JUN","JUL","AGO","SEP","OCT","NOV","DIC"]
STOP_WORDS    = {"MÁXIMO","MAXIMO","MÍNIMO","MINIMO","MEDIO"}
EXCEL_NAME    = "caudal.xlsx"
DPI           = 150
PERM_VERANO   = 70    # % permanencia → umbral verano
PERM_INVIERNO = 30    # % permanencia → umbral invierno

# Carpetas a omitir dentro de cada río (sin datos de caudal)
OMITIR_SUBCARPETAS = {
    "PTE VARIANTE (SIN DATOS)",
    "VOLADERO (SIN DATOS)",
    "CHORRERA DEL INDIO",
    "ABAJO (S)",
    "TOCHE (S)",
}

# ── Verificación de ruta ──────────────────────────────────────────────────────
if not RAIZ_TRIBUTARIOS.exists():
    print(f"⚠  La carpeta '{RAIZ_TRIBUTARIOS}' no existe.")
    print("   Ajusta RAIZ_TRIBUTARIOS en esta celda y vuelve a ejecutar.")
else:
    html_files = list(RAIZ_TRIBUTARIOS.rglob("Informe*.html"))
    xlsx_risaralda = list((RAIZ_TRIBUTARIOS / "RIO RISARALDA" / "CAUDAL").glob("*.xlsx")) \
                     if (RAIZ_TRIBUTARIOS / "RIO RISARALDA" / "CAUDAL").exists() else []
    print(f"✅ Raíz encontrada: {RAIZ_TRIBUTARIOS.resolve()}")
    print(f"   Informes HTML  : {len(html_files)} archivos")
    print(f"   XLSX Risaralda : {len(xlsx_risaralda)} archivos")
    for f in sorted(html_files):
        print(f"   • {f.relative_to(RAIZ_TRIBUTARIOS)}")


⚠  La carpeta 'Tributarios' no existe.
   Ajusta RAIZ_TRIBUTARIOS en esta celda y vuelve a ejecutar.


## Celda 2 · Extractor HTML → Excel por estación
Lee cada `Informe.html` del portal CVC, extrae las tablas de **Caudal Promedio Diario**
y guarda un `caudal.xlsx` con una hoja por año dentro de la misma carpeta de la estación.

> Las carpetas marcadas `(SIN DATOS)` se omiten automáticamente.


In [2]:
def limpiar_valor(v: str):
    v = v.strip()
    if v in ("***<", "***", "<", "", "—", "-"):
        return None
    try:
        return float(v.replace(",", "."))
    except ValueError:
        return None


def extraer_año_html(spans, data_start, data_end):
    block = spans[data_start:data_end]
    max_off = next(
        (i for i, t in enumerate(block) if t.upper() in ("MÁXIMO", "MAXIMO")),
        len(block)
    )
    datos = block[:max_off]
    filas = {}
    i = 0
    while i < len(datos):
        tok = datos[i].strip()
        if tok.upper() in STOP_WORDS:
            i += 1; continue
        try:
            dia = int(tok)
            if not 1 <= dia <= 31:
                i += 1; continue
        except ValueError:
            i += 1; continue
        vals, j = [], i + 1
        while j < len(datos):
            cand = datos[j].strip()
            if cand.upper() in STOP_WORDS:
                break
            try:
                nd = int(cand)
                if 1 <= nd <= 31 and vals:
                    break
            except ValueError:
                pass
            is_val = False
            try:
                float(cand.replace(",", ".")); is_val = True
            except ValueError:
                if cand in ("***<", "***", "<"):
                    is_val = True
            if is_val:
                vals.append(limpiar_valor(cand)); j += 1
            else:
                j += 1
        filas[dia] = (vals + [None]*12)[:12]
        i = j

    rows = [[dia] + filas[dia] for dia in sorted(filas.keys())]

    def _extraer_resumen(kw_list):
        off = next((i for i, t in enumerate(block) if t.upper() in kw_list), None)
        if off is None:
            return [None]*12
        raw = block[off+1:off+30]
        tmp, k = [], 0
        while len(tmp) < 12 and k < len(raw)-1:
            try:
                tmp.append(float(raw[k].replace(",", "."))); k += 2
            except Exception:
                k += 1
        return (tmp + [None]*12)[:12]

    max_vals = [limpiar_valor(v) for v in block[max_off+1:max_off+13]]
    max_vals = (max_vals + [None]*12)[:12]
    min_vals = _extraer_resumen({"MÍNIMO","MINIMO"})
    med_vals = _extraer_resumen({"MEDIO"})
    return rows, max_vals, min_vals, med_vals


def escribir_hoja_xlsx(ws, titulo_estacion, año, rows, max_vals, min_vals, med_vals):
    borde = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin"),
    )
    ws.merge_cells("A1:M1")
    c = ws["A1"]
    c.value = f"Caudal Promedio Diario  |  Estación: {titulo_estacion}  |  Año: {año}  |  Unidad: m³/s"
    c.font  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    c.fill  = PatternFill("solid", fgColor="1D4E8F")
    c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 20

    for col, h in enumerate(["DÍA"] + MESES, 1):
        c = ws.cell(row=2, column=col, value=h)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=9)
        c.fill = PatternFill("solid", fgColor="2E75B6")
        c.alignment = Alignment(horizontal="center")
        c.border = borde

    for r, row in enumerate(rows, 3):
        for col, val in enumerate(row, 1):
            c = ws.cell(row=r, column=col)
            if col == 1:
                c.value = int(val); c.fill = PatternFill("solid", fgColor="EBF3FB")
                c.alignment = Alignment(horizontal="center")
                c.font = Font(bold=True, name="Arial", size=9)
            elif val is None:
                c.value = "S/D"; c.font = Font(name="Arial", size=9, italic=True, color="999999")
                c.alignment = Alignment(horizontal="center")
            else:
                c.value = val; c.number_format = "#,##0.00"
                c.font = Font(name="Arial", size=9)
                c.alignment = Alignment(horizontal="right")
            c.border = borde

    sep = len(rows) + 3
    for offset, (etiq, vals, bg) in enumerate([
        ("MÁXIMO", max_vals, "FFC000"),
        ("MÍNIMO", min_vals, "70AD47"),
        ("MEDIO",  med_vals, "4472C4"),
    ]):
        r = sep + offset
        c = ws.cell(row=r, column=1, value=etiq)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=9)
        c.fill = PatternFill("solid", fgColor=bg)
        c.alignment = Alignment(horizontal="center"); c.border = borde
        for col, v in enumerate(vals, 2):
            cell = ws.cell(row=r, column=col, value=v)
            cell.font = Font(bold=True, name="Arial", size=9, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor=bg)
            cell.alignment = Alignment(horizontal="right")
            if v is not None:
                cell.number_format = "#,##0.00"
            cell.border = borde

    ws.column_dimensions["A"].width = 6
    for col in range(2, 14):
        ws.column_dimensions[get_column_letter(col)].width = 10
    ws.freeze_panes = "B3"


def procesar_html(html_path: Path):
    nombre_carpeta = html_path.parent.name.upper()
    if nombre_carpeta in OMITIR_SUBCARPETAS:
        print(f"  ⏭  {html_path.parent.relative_to(RAIZ_TRIBUTARIOS)} — omitida")
        return None

    # Título para el Excel: "RIO / ESTACIÓN"
    partes = html_path.parts
    try:
        idx_raiz = partes.index(RAIZ_TRIBUTARIOS.name)
        subtitulo = " / ".join(partes[idx_raiz+1:-1])
    except (ValueError, IndexError):
        subtitulo = nombre_carpeta

    output = html_path.parent / EXCEL_NAME
    print(f"📄 {subtitulo}")

    with open(html_path, encoding="utf-8", errors="replace") as f:
        soup = BeautifulSoup(f, "lxml")

    spans = [s.get_text(strip=True) for s in soup.find_all("span")]
    año_indices = [
        (i, spans[i+1])
        for i, t in enumerate(spans)
        if t == "AÑO" and i+1 < len(spans) and spans[i+1].isdigit()
    ]
    if not año_indices:
        print(f"   ⚠  Sin tablas de caudal — omitida")
        return None

    print(f"   → {len(año_indices)} años: {', '.join(a for _, a in año_indices)}")
    wb = Workbook(); wb.remove(wb.active)

    for k, (idx, año) in enumerate(año_indices):
        data_start = idx + 15
        data_end   = año_indices[k+1][0] if k+1 < len(año_indices) else len(spans)
        rows, max_v, min_v, med_v = extraer_año_html(spans, data_start, data_end)
        ws = wb.create_sheet(title=str(año))
        escribir_hoja_xlsx(ws, subtitulo, año, rows, max_v, min_v, med_v)
        print(f"   ✔  {año} — {len(rows)} días")

    wb.save(output)
    print(f"   ✅ → {output.relative_to(RAIZ_TRIBUTARIOS)}\n")
    return output


# ── Ejecutar ──────────────────────────────────────────────────────────────────
html_files = sorted(RAIZ_TRIBUTARIOS.rglob("Informe*.html"))
print(f"🗂  {len(html_files)} archivos HTML encontrados\n")
excels_generados = []
for html_path in html_files:
    result = procesar_html(html_path)
    if result:
        excels_generados.append(result)

print(f"\n✅ Extracción completada — {len(excels_generados)} archivos Excel generados.")


🗂  0 archivos HTML encontrados


✅ Extracción completada — 0 archivos Excel generados.


## Celda 3 · Curvas de duración de caudal
Por cada `caudal.xlsx` genera:
- `curva_duracion.png` — gráfica con zonas Verano / Transición / Invierno
- `umbrales_caudal.txt` — umbrales calculados (percentiles 30 % / 70 %)


In [3]:
def leer_caudales_excel(excel_path: Path) -> np.ndarray:
    wb = load_workbook(excel_path, data_only=True)
    valores = []
    for sheet_name in wb.sheetnames:
        try:
            int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            for val in row[1:13]:
                if isinstance(val, (int, float)) and not np.isnan(val):
                    valores.append(float(val))
    return np.array(valores)


def calcular_umbrales(caudales: np.ndarray):
    datos = np.sort(caudales)[::-1]
    perm  = np.arange(1, len(datos)+1) / len(datos) * 100
    u_v   = float(np.interp(PERM_VERANO,   perm, datos))
    u_i   = float(np.interp(PERM_INVIERNO, perm, datos))
    return perm, datos, u_v, u_i


def graficar_curva_duracion(nombre: str, perm, caudales, u_v, u_i, out_path: Path):
    y_max = caudales.max()
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor("#F7F9FC"); ax.set_facecolor("#F7F9FC")

    ax.axvspan(0,              PERM_INVIERNO, alpha=0.08, color="#2563EB")
    ax.axvspan(PERM_INVIERNO,  PERM_VERANO,  alpha=0.08, color="#16A34A")
    ax.axvspan(PERM_VERANO,    100,           alpha=0.08, color="#F59E0B")

    ax.plot(perm, caudales, color="#1E3A5F", linewidth=1.8, zorder=4)
    ax.axvline(PERM_INVIERNO, color="#2563EB", linewidth=1.4, linestyle="--", zorder=5)
    ax.axvline(PERM_VERANO,   color="#F59E0B", linewidth=1.4, linestyle="--", zorder=5)
    ax.axhline(u_i, color="#2563EB", linewidth=1.0, linestyle=":", alpha=0.7, zorder=5)
    ax.axhline(u_v, color="#F59E0B", linewidth=1.0, linestyle=":", alpha=0.7, zorder=5)

    xt_i = min(PERM_INVIERNO + 2, 95)
    xt_v = min(PERM_VERANO + 2, 95)
    ax.annotate(f"Invierno ≥ {u_i:,.1f} m³/s\n(< {PERM_INVIERNO}%)",
                xy=(PERM_INVIERNO, u_i), xytext=(xt_i, u_i*1.08),
                fontsize=8.5, color="#2563EB", fontweight="bold",
                arrowprops=dict(arrowstyle="-", color="#2563EB", lw=0.8))
    ax.annotate(f"Verano ≤ {u_v:,.1f} m³/s\n(> {PERM_VERANO}%)",
                xy=(PERM_VERANO, u_v), xytext=(xt_v, u_v*1.08),
                fontsize=8.5, color="#B45309", fontweight="bold",
                arrowprops=dict(arrowstyle="-", color="#B45309", lw=0.8))

    ax.text(PERM_INVIERNO/2,          y_max*0.92, "INVIERNO",    ha="center", fontsize=9, color="#2563EB", fontweight="bold", alpha=0.7)
    ax.text((PERM_INVIERNO+PERM_VERANO)/2, y_max*0.92, "TRANSICIÓN", ha="center", fontsize=9, color="#16A34A", fontweight="bold", alpha=0.7)
    ax.text((PERM_VERANO+100)/2,      y_max*0.92, "VERANO",     ha="center", fontsize=9, color="#B45309", fontweight="bold", alpha=0.7)

    ax.set_xlim(0, 100); ax.set_ylim(0, y_max*1.05)
    ax.set_xlabel("Porcentaje de permanencia (%)", fontsize=10, labelpad=8)
    ax.set_ylabel("Caudal (m³/s)", fontsize=10, labelpad=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.1f}"))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.grid(axis="both", linestyle="--", alpha=0.35, zorder=0)
    ax.set_title(
        f"Curva de Duración de Caudales — {nombre}\n"
        f"Datos diarios  |  n = {len(caudales):,} registros",
        fontsize=12, fontweight="bold", pad=14, color="#1A1A2E")
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


def guardar_umbrales_txt(nombre: str, caudales: np.ndarray, u_v, u_i, out_path: Path):
    lineas = [
        f"Estación : {nombre}",
        f"Registros: {len(caudales):,} caudales diarios",
        "",
        "── Umbrales (percentil 70% / 30%) ──────────────────",
        f"  INVIERNO  : caudal ≥ {u_i:,.2f} m³/s  (permanencia < {PERM_INVIERNO}%)",
        f"  TRANSICIÓN: {u_v:,.2f} – {u_i:,.2f} m³/s",
        f"  VERANO    : caudal ≤ {u_v:,.2f} m³/s  (permanencia > {PERM_VERANO}%)",
        "",
        "── Estadísticas generales ───────────────────────────",
        f"  Mínimo   : {caudales.min():,.2f} m³/s",
        f"  Máximo   : {caudales.max():,.2f} m³/s",
        f"  Promedio : {caudales.mean():,.2f} m³/s",
        f"  Mediana  : {np.median(caudales):,.2f} m³/s",
    ]
    out_path.write_text("\n".join(lineas), encoding="utf-8")


# ── Ejecutar ──────────────────────────────────────────────────────────────────
excels = sorted(RAIZ_TRIBUTARIOS.rglob(EXCEL_NAME))
print(f"🗂  {len(excels)} archivos caudal.xlsx encontrados\n")
print(f"  {'Estación':<35} | {'n datos':>8} | {'Verano ≤':>10} | {'Invierno ≥':>11}")
print(f"  {'-'*35} | {'-'*8} | {'-'*10} | {'-'*11}")

for excel_path in excels:
    nombre = " / ".join(excel_path.parts[
        list(excel_path.parts).index(RAIZ_TRIBUTARIOS.name)+1:-1])
    caudales = leer_caudales_excel(excel_path)
    if len(caudales) == 0:
        print(f"  ⚠  {nombre:<35} — sin datos"); continue

    perm, caud_ord, u_v, u_i = calcular_umbrales(caudales)
    graficar_curva_duracion(nombre, perm, caud_ord, u_v, u_i,
                            excel_path.parent / "curva_duracion.png")
    guardar_umbrales_txt(nombre, caudales, u_v, u_i,
                         excel_path.parent / "umbrales_caudal.txt")
    print(f"  ✔  {nombre:<35} | {len(caudales):>8,} | {u_v:>10.2f} | {u_i:>11.2f}")

print("\n✅ Curvas de duración generadas.")


🗂  0 archivos caudal.xlsx encontrados

  Estación                            |  n datos |   Verano ≤ |  Invierno ≥
  ----------------------------------- | -------- | ---------- | -----------

✅ Curvas de duración generadas.


## Celda 4 · Caudal promedio mensual por quinquenio
Genera `quinquenios_caudal.png` en cada carpeta de estación.
Los quinquenios son: 2010–2014 / 2015–2019 / 2020–2026.


In [4]:
QUINQUENIOS = {
    "2010 – 2014": range(2010, 2015),
    "2015 – 2019": range(2015, 2020),
    "2020 – 2026": range(2020, 2027),
}
COLORES_Q = {
    "2010 – 2014": "#2563EB",
    "2015 – 2019": "#16A34A",
    "2020 – 2026": "#DC2626",
}


def leer_medios_mensuales(excel_path: Path) -> dict:
    wb = load_workbook(excel_path, data_only=True)
    datos = {}
    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            if str(row[0]).strip().upper() == "MEDIO":
                vals = [v if isinstance(v, (int, float)) else None for v in row[1:13]]
                datos[año] = vals
                break
    return datos


def promedio_quinquenio(datos, años):
    result = []
    for mes in range(12):
        vals = [datos[a][mes] for a in años if a in datos and datos[a][mes] is not None]
        result.append(round(np.mean(vals), 2) if vals else None)
    return result


def graficar_quinquenios_trib(nombre: str, datos: dict, out_path: Path):
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor("#F7F9FC"); ax.set_facecolor("#F7F9FC")
    tiene_datos = False

    for label, años in QUINQUENIOS.items():
        prom = promedio_quinquenio(datos, años)
        xs = [i for i, v in enumerate(prom) if v is not None]
        ys = [v for v in prom if v is not None]
        if not ys:
            continue
        tiene_datos = True
        color = COLORES_Q[label]
        ax.plot(xs, ys, color=color, linewidth=2.2, marker="o", markersize=6,
                markerfacecolor="white", markeredgewidth=2, markeredgecolor=color,
                label=label, zorder=4)
        for x, y in zip(xs, ys):
            ax.annotate(f"{y:,.1f}", xy=(x, y), xytext=(0, 9),
                        textcoords="offset points", ha="center", fontsize=7,
                        color=color, fontweight="bold")

    if not tiene_datos:
        plt.close(fig); return

    ax.set_xticks(range(12)); ax.set_xticklabels(MESES, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.1f}"))
    ax.tick_params(axis="y", labelsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0); ax.set_axisbelow(True)
    sns.despine(ax=ax)
    ax.set_title(f"Caudal Promedio Mensual por Quinquenio\n{nombre}",
                 fontsize=13, fontweight="bold", pad=14, color="#1A1A2E")
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.set_xlabel("Mes", fontsize=10, labelpad=8)
    ax.legend(title="Quinquenio", fontsize=10, title_fontsize=10,
              framealpha=0.8, loc="best")
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


# ── Ejecutar ──────────────────────────────────────────────────────────────────
excels = sorted(RAIZ_TRIBUTARIOS.rglob(EXCEL_NAME))
print(f"🗂  {len(excels)} archivos encontrados\n")

for excel_path in excels:
    nombre = " / ".join(excel_path.parts[
        list(excel_path.parts).index(RAIZ_TRIBUTARIOS.name)+1:-1])
    datos = leer_medios_mensuales(excel_path)
    if not datos:
        print(f"  ⚠  {nombre} — sin datos"); continue
    out = excel_path.parent / "quinquenios_caudal.png"
    graficar_quinquenios_trib(nombre, datos, out)
    print(f"  ✔  {nombre}")

print("\n✅ Gráficas de quinquenios generadas.")


🗂  0 archivos encontrados


✅ Gráficas de quinquenios generadas.


## Celda 5 · Caudal promedio anual
Genera `promedio_anual.png` en cada carpeta de estación, con barra por año
y línea de promedio del período.


In [5]:
AÑO_INICIO = 2010
AÑO_FIN    = 2026


def leer_promedios_anuales(excel_path: Path) -> dict:
    wb = load_workbook(excel_path, data_only=True)
    resultados = {}
    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        if not (AÑO_INICIO <= año <= AÑO_FIN):
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            if str(row[0]).strip().upper() == "MEDIO":
                vals = [v for v in row[1:13] if isinstance(v, (int, float))]
                if vals:
                    resultados[año] = round(np.mean(vals), 2)
                break
    return resultados


def graficar_promedio_anual_trib(nombre: str, promedios: dict, out_path: Path):
    años    = sorted(promedios.keys())
    valores = [promedios[a] for a in años]
    norm    = plt.Normalize(min(valores), max(valores))
    colores = plt.cm.YlGnBu(norm(valores))

    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor("#F7F9FC"); ax.set_facecolor("#F7F9FC")

    bars = ax.bar(años, valores, color=colores, edgecolor="white",
                  linewidth=0.7, width=0.65, zorder=3)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(valores)*0.012,
                f"{val:,.1f}", ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="#333333")

    prom_total = np.mean(valores)
    ax.axhline(prom_total, color="#E63946", linewidth=1.4, linestyle="--", zorder=4,
               label=f"Promedio período: {prom_total:,.1f} m³/s")

    ax.set_xticks(años)
    ax.set_xticklabels([str(a) for a in años], rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.1f}"))
    ax.tick_params(axis="y", labelsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.45, zorder=0); ax.set_axisbelow(True)
    sns.despine(ax=ax)
    ax.set_title(f"Caudal Promedio Anual\n{nombre}",
                 fontsize=13, fontweight="bold", pad=14, color="#1A1A2E")
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.set_xlabel("Año", fontsize=10, labelpad=8)
    ax.legend(fontsize=9, framealpha=0.7)
    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


# ── Ejecutar ──────────────────────────────────────────────────────────────────
excels = sorted(RAIZ_TRIBUTARIOS.rglob(EXCEL_NAME))
print(f"🗂  {len(excels)} archivos encontrados\n")

for excel_path in excels:
    nombre = " / ".join(excel_path.parts[
        list(excel_path.parts).index(RAIZ_TRIBUTARIOS.name)+1:-1])
    promedios = leer_promedios_anuales(excel_path)
    if not promedios:
        print(f"  ⚠  {nombre} — sin datos en {AÑO_INICIO}-{AÑO_FIN}"); continue
    out = excel_path.parent / "promedio_anual.png"
    graficar_promedio_anual_trib(nombre, promedios, out)
    años = sorted(promedios.keys())
    print(f"  ✔  {nombre}  ({años[0]}–{años[-1]})")

print("\n✅ Gráficas de promedio anual generadas.")


🗂  0 archivos encontrados


✅ Gráficas de promedio anual generadas.


## Celda 6 · Río Risaralda — XLSX propios (CMáx / CMed / CMín)

El Risaralda tiene datos de fuente diferente al portal CVC (posiblemente CARDER / EPM):
dos estaciones (`Rio Risaralda EHT` y `Casa de Máquinas`) con columnas
CMáx, CMed, CMín por mes para 2025–2026.

> ⚠ **Limitación:** con solo ~1.5 años de datos no es posible construir una curva de
> duración estadísticamente representativa. Se generan en cambio:
> - Gráfica de variación mensual CMed / CMáx / CMín para 2025
> - Tabla resumen de estadísticas disponibles

Los archivos de salida se guardan en `RIO RISARALDA/CAUDAL/Gráficas/`.


In [6]:
def leer_xlsx_risaralda(xlsx_path: Path) -> pd.DataFrame:
    """Lee un xlsx con formato CMáx/CMed/CMín y devuelve DataFrame tidy."""
    wb = load_workbook(xlsx_path, data_only=True)
    registros = []

    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        rows = list(ws.iter_rows(min_row=1, values_only=True))
        if len(rows) < 4:
            continue

        # Fila 2: meses (con celdas fusionadas → repetir mes)
        # Fila 3: CMáx / CMed / CMín (sub-columnas)
        # Fila 4+: datos

        # Reconstruir encabezado de columnas
        fila_mes  = rows[1]   # índice 1 = fila 2
        fila_tipo = rows[2]   # índice 2 = fila 3

        # Propagar mes en celdas None (celdas fusionadas)
        meses_col = []
        mes_actual = None
        for v in fila_mes[1:]:   # saltar columna DÍA
            if v is not None:
                mes_actual = v
            meses_col.append(mes_actual)

        tipos_col = [str(v).strip() if v else "" for v in fila_tipo[1:]]

        for row in rows[3:]:
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            for col_idx, (mes, tipo) in enumerate(zip(meses_col, tipos_col)):
                if mes is None or tipo not in ("CMáx", "CMed", "CMín"):
                    continue
                val = row[col_idx + 1]
                if isinstance(val, (int, float)):
                    mes_num = MESES.index(mes) + 1 if mes in MESES else None
                    if mes_num:
                        try:
                            fecha = pd.Timestamp(year=año, month=mes_num, day=dia)
                            registros.append({"FECHA": fecha, "TIPO": tipo, "CAUDAL": float(val)})
                        except ValueError:
                            pass
    return pd.DataFrame(registros)


def graficar_risaralda(nombre_estacion: str, df: pd.DataFrame, out_path: Path):
    """Gráfica de variación mensual CMed/CMáx/CMín para los años disponibles."""
    if df.empty:
        print(f"  ⚠  {nombre_estacion} — sin datos"); return

    # Agrupar por mes y tipo
    df["MES"]  = df["FECHA"].dt.month
    df["AÑO"]  = df["FECHA"].dt.year
    años_disp  = sorted(df["AÑO"].unique())

    # Promediar CMed mensual por año
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor("#F7F9FC"); ax.set_facecolor("#F7F9FC")

    colores_año = ["#1D4E8F", "#DC2626", "#16A34A", "#B45309"]
    for i, año in enumerate(años_disp):
        sub = df[df["AÑO"] == año]
        med = sub[sub["TIPO"] == "CMed"].groupby("MES")["CAUDAL"].mean()
        mx  = sub[sub["TIPO"] == "CMáx"].groupby("MES")["CAUDAL"].mean()
        mn  = sub[sub["TIPO"] == "CMín"].groupby("MES")["CAUDAL"].mean()

        meses_idx = [m-1 for m in med.index]  # 0-based para etiquetas
        col = colores_año[i % len(colores_año)]

        if not med.empty:
            ax.plot(meses_idx, med.values, color=col, linewidth=2.2,
                    marker="o", markersize=6, markerfacecolor="white",
                    markeredgewidth=2, markeredgecolor=col, label=f"CMed {año}", zorder=4)
            if not mx.empty and not mn.empty:
                mx_vals = [mx.get(m+1, np.nan) for m in meses_idx]
                mn_vals = [mn.get(m+1, np.nan) for m in meses_idx]
                ax.fill_between(meses_idx, mn_vals, mx_vals,
                                alpha=0.12, color=col, label=f"CMín–CMáx {año}")

    ax.set_xticks(range(12)); ax.set_xticklabels(MESES, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.1f}"))
    ax.tick_params(axis="y", labelsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0); ax.set_axisbelow(True)
    sns.despine(ax=ax)
    ax.set_title(
        f"Variación Mensual de Caudal — {nombre_estacion}\n"
        f"Línea = CMed  |  Banda = CMín–CMáx  |  Período: {años_disp[0]}–{años_disp[-1]}",
        fontsize=12, fontweight="bold", pad=14, color="#1A1A2E")
    ax.set_ylabel("Caudal (m³/s)", fontsize=10, labelpad=8)
    ax.set_xlabel("Mes", fontsize=10, labelpad=8)
    ax.legend(fontsize=9, framealpha=0.8, loc="best")

    nota = ("⚠ Serie corta (2025–2026): insuficiente para curva de duración representativa. "
            "Se recomienda solicitar series históricas a CARDER.")
    fig.text(0.5, -0.02, nota, ha="center", fontsize=8, color="#B45309", style="italic")

    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)


# ── Ejecutar ──────────────────────────────────────────────────────────────────
carpeta_rda = RAIZ_TRIBUTARIOS / "RIO RISARALDA" / "CAUDAL"

if not carpeta_rda.exists():
    print(f"⚠  No se encontró la carpeta: {carpeta_rda}")
    print("   Verifica que la ruta RAIZ_TRIBUTARIOS sea correcta.")
else:
    out_dir = carpeta_rda.parent / "Gráficas"
    out_dir.mkdir(exist_ok=True)

    xlsx_files = sorted(carpeta_rda.glob("*.xlsx"))
    print(f"🗂  {len(xlsx_files)} archivos XLSX encontrados en {carpeta_rda.relative_to(RAIZ_TRIBUTARIOS)}\n")

    tablas_resumen = {}
    for xlsx_path in xlsx_files:
        nombre = xlsx_path.stem
        df_rda = leer_xlsx_risaralda(xlsx_path)

        if df_rda.empty:
            print(f"  ⚠  {nombre} — sin datos extraídos"); continue

        n_total = len(df_rda[df_rda["TIPO"] == "CMed"])
        años    = sorted(df_rda["FECHA"].dt.year.unique())
        print(f"  ✔  {nombre}")
        print(f"     Años disponibles: {años}")
        print(f"     Registros CMed  : {n_total}")

        out_png = out_dir / f"caudal_mensual_{nombre}.png"
        graficar_risaralda(nombre, df_rda, out_png)
        print(f"     → {out_png.relative_to(RAIZ_TRIBUTARIOS)}")

        # Resumen estadístico
        med = df_rda[df_rda["TIPO"] == "CMed"]["CAUDAL"]
        tablas_resumen[nombre] = {
            "Años"    : f"{años[0]}–{años[-1]}",
            "n CMed"  : n_total,
            "Mín"     : f"{med.min():.2f}",
            "Máx"     : f"{med.max():.2f}",
            "Promedio": f"{med.mean():.2f}",
            "Mediana" : f"{med.median():.2f}",
        }
        print()

    if tablas_resumen:
        print("── Resumen estadístico (CMed) ────────────────────────────")
        df_res = pd.DataFrame(tablas_resumen).T
        print(df_res.to_string())
        print()
        print("⚠  Nota: con series de 1–2 años no se calculan umbrales V/T/I.")
        print("   Para análisis hidrológico representativo se recomienda gestionar")
        print("   series históricas ante CARDER (cuenca Risaralda, fuera de CVC).")

print("\n✅ Análisis Risaralda completado.")


⚠  No se encontró la carpeta: Tributarios\RIO RISARALDA\CAUDAL
   Verifica que la ruta RAIZ_TRIBUTARIOS sea correcta.

✅ Análisis Risaralda completado.
